# Minimal Baseline: Random Forest Fight Detection

This notebook is intentionally minimal and self-contained.

- Features used: `inter_animal_distance`, `speed1`, `speed2`
- No temporal windowing
- Train on `gt1` only
- Apply to `gt2` and compare to manual annotations

In [ ]:
# Start: import all libraries used for feature extraction, modeling, metrics, and plots.
# Beginner note: numpy/pandas handle data; matplotlib makes plots; sklearn provides Random Forest + metrics.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# RandomForestClassifier is the model; metric functions evaluate predictions.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from QuantBio_functions import ensure_quant_beh_data_from_zenodo

In [ ]:
# Next: define fixed experiment settings and ensure required data files exist locally (Zenodo download if needed).
# Beginner note: this is the only external helper call, used to keep file setup reliable for everyone.
# ----------------------
# Hard-coded minimal settings
# ----------------------
FPS = 60  # Frames per second of the source video
TRAIN_GT = "gt1"  # Dataset used to train the model
EVAL_GT = "gt2"   # Separate dataset used only for evaluation
THRESHOLD = 0.5  # Probability cutoff: >= threshold => predicted fight

# Random Forest hyperparameters (simple defaults to start).
RF_PARAMS = {
    "n_estimators": 200,  # Number of trees; more trees usually = smoother predictions
    "random_state": 42,   # Fix seed so results are repeatable
    "class_weight": "balanced",  # Helps when class 1 (fight) is less frequent
    "n_jobs": -1,  # Use all CPU cores
}



download_status = ensure_quant_beh_data_from_zenodo(verbose=True)
DATA_ROOT = download_status["data_dir"]
print("Using DATA_ROOT:", DATA_ROOT)

# Fail early with a clear message if required files are still missing.
if len(download_status["missing_required"]) > 0:
    raise FileNotFoundError(
        "Some required data files are still missing: " + ", ".join(download_status["missing_required"])
)

In [ ]:
# Next: create helper functions to compute minimal features and merge them with manual labels.
# Beginner note: these functions isolate preprocessing logic so training code stays clean.
def compute_minimal_features(traj_df, fps=60):
    # Extract x/y positions for both fish as numeric arrays.
    fish1 = traj_df[["x1", "y1"]].to_numpy()
    fish2 = traj_df[["x2", "y2"]].to_numpy()

    # Replace missing values with 0 to avoid NaN issues in math operations.
    fish1 = np.nan_to_num(fish1, nan=0.0)
    fish2 = np.nan_to_num(fish2, nan=0.0)

    # Feature 1: Euclidean distance between fish positions per frame.
    inter_animal_distance = np.linalg.norm(fish1 - fish2, axis=1)

    # Features 2-3: per-frame speed for each fish (pixels/frame converted to pixels/second).
    # np.diff computes frame-to-frame displacement; prepend keeps output length equal to input length.
    speed1 = np.linalg.norm(np.diff(fish1, axis=0, prepend=fish1[0:1]), axis=1) * fps
    speed2 = np.linalg.norm(np.diff(fish2, axis=0, prepend=fish2[0:1]), axis=1) * fps

    # Extra example features: raw absolute coordinates.
    # These are often less informative across recordings because absolute position can shift with camera/setup.
    x1 = fish1[:, 0]
    y1 = fish1[:, 1]
    x2 = fish2[:, 0]
    y2 = fish2[:, 1]

    # Return one row per frame with all model input features.
    return pd.DataFrame({
        "frame": np.arange(len(traj_df), dtype=int),
        "inter_animal_distance": inter_animal_distance,
        "speed1": speed1,
        "speed2": speed2,
        "x1": x1,
        "y1": y1,
        "x2": x2,
        "y2": y2,
    })

def load_gt_as_table(gt_id, data_root, fps=60):
    # Load trajectories (positions) and manual labels from CSV files.
    traj_df = pd.read_csv(os.path.join(data_root, f"{gt_id}_trajectories.csv"))
    label_df = pd.read_csv(os.path.join(data_root, f"{gt_id}_manual_labeled_fights.csv"))

    # Build features and align with labels by frame index.
    feat_df = compute_minimal_features(traj_df, fps=fps)
    merged = pd.merge(feat_df, label_df[["frame", "label"]], on="frame", how="inner")

    # Convert labels to strict binary values: 0 (no fight), 1 (fight).
    merged["label"] = (merged["label"].astype(int) > 0).astype(int)
    return merged
    

In [ ]:
# Next: load train/eval tables, fit the random forest, and inspect basic dataset/feature statistics.
# Beginner note: keep train and eval data separate to test generalization honestly.
train_df = load_gt_as_table(TRAIN_GT, DATA_ROOT, fps=FPS)
eval_df = load_gt_as_table(EVAL_GT, DATA_ROOT, fps=FPS)

# Define model inputs.
# The first three are behavior-oriented; the raw x/y coordinates are included as examples that may be less informative.
FEATURES = ["inter_animal_distance", "speed1", "speed2", "x1", "y1", "x2", "y2"]

print(f"Training on {TRAIN_GT}: {len(train_df)} rows")
print(f"Evaluating on {EVAL_GT}: {len(eval_df)} rows")

# Quick look at first rows to verify columns and value ranges.
print("\nTrain preview:")
display(train_df.head())
print("\nEval preview:")
display(eval_df.head())

# Build feature matrix X and target vector y for training.
X_train = train_df[FEATURES]
y_train = train_df["label"]

# Create and fit the Random Forest model.
clf = RandomForestClassifier(**RF_PARAMS)
clf.fit(X_train, y_train)


In [ ]:
# Next: inspect which input features the trained Random Forest uses most.
# Beginner note: higher importance means the model relied more on that feature for splits across trees.
feature_importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": clf.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Feature importance (higher = more used by the model):")
display(feature_importance_df)

plt.figure(figsize=(7, 3.5))
plt.bar(feature_importance_df["feature"], feature_importance_df["importance"])
plt.ylabel("Importance")
plt.xlabel("Feature")
plt.title(f"Random Forest feature importance (train={TRAIN_GT})")
plt.tight_layout()
plt.show()

## Quick sanity check on the training dataset (gt1)

> Why do this before gt2?
- This confirms the model learned the training labels and the pipeline is working end-to-end.
- If training performance were unexpectedly low, we would debug feature extraction or label alignment before testing generalization.

Important interpretation:
- Very high train performance is common for flexible models like Random Forests.
- The real test is performance on unseen data (`gt2`), which comes next.

In [ ]:
# Next: evaluate model behavior on gt1 (the same data used for training).
# Beginner note: this is a sanity check, not a generalization test.
X_train_eval = train_df[FEATURES]
y_train_eval = train_df["label"].to_numpy()

# Predicted probability of class 1 (fight) on training data.
prob_train = clf.predict_proba(X_train_eval)[:, 1]
pred_train = (prob_train >= THRESHOLD).astype(int)

# Compute standard metrics on training data.
acc_train = accuracy_score(y_train_eval, pred_train)
prec_train = precision_score(y_train_eval, pred_train, zero_division=0)
rec_train = recall_score(y_train_eval, pred_train, zero_division=0)
f1_train = f1_score(y_train_eval, pred_train, zero_division=0)
tn_train, fp_train, fn_train, tp_train = confusion_matrix(y_train_eval, pred_train, labels=[0, 1]).ravel()


# Keep train metrics in a one-row table; we will compare to gt2 after gt2 inference.
train_metrics_df = pd.DataFrame([
    {
        "dataset": TRAIN_GT,
        "accuracy": acc_train,
        "precision": prec_train,
        "recall": rec_train,
        "f1": f1_train,
        "tn": tn_train,
        "fp": fp_train,
        "fn": fn_train,
        "tp": tp_train,
    }
])
display(train_metrics_df)

# Build per-frame outputs for a train-set overlay plot.
train_out = train_df[["frame", "label"]].copy()
train_out["fight_probability"] = prob_train
train_out["predicted_label"] = pred_train

# Overlay plot on a slice of gt1.
start_train = 0
end_train = min(4000, len(train_out))
vt = train_out.iloc[start_train:end_train].copy()

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(vt["frame"], vt["fight_probability"], lw=1)
axes[0].axhline(THRESHOLD, color="black", linestyle="--", label="threshold")
axes[0].set_ylabel("P(fight)")
axes[0].legend(loc="upper right")

axes[1].plot(vt["frame"], vt["label"] + 0.1, label="manual", lw=1)
axes[1].plot(vt["frame"], vt["predicted_label"], label="predicted", lw=1, alpha=0.8)
axes[1].set_ylabel("Label")
axes[1].set_xlabel("Frame")
axes[1].legend(loc="upper right")

plt.suptitle(f"Training-set check: train={TRAIN_GT}")
plt.tight_layout()
plt.show()

In [ ]:
# Next: run inference on eval data and convert probabilities to labels using the threshold.
# Beginner note: we first generate predictions, then analyze metrics afterwards.
X_eval = eval_df[FEATURES]
y_eval = eval_df["label"].to_numpy()

# predict_proba returns [P(class=0), P(class=1)] for each row; we keep P(fight=1).
prob_eval = clf.predict_proba(X_eval)[:, 1]

# Convert probabilities to binary predictions using the current threshold.
pred_eval = (prob_eval >= THRESHOLD).astype(int)

# Keep per-frame outputs for immediate visualization and later metric analysis.
eval_out = eval_df[["frame", "label"]].copy()
eval_out["fight_probability"] = prob_eval
eval_out["predicted_label"] = pred_eval

print(f"Inference complete on {EVAL_GT} using threshold={THRESHOLD}")
print("\nPrediction preview:")
display(eval_out.head(10))

In [ ]:
# Next: visualize a frame slice comparing predicted probabilities and binary labels to manual labels.
# Beginner note: this helps diagnose where the model is uncertain or systematically off.
# Quick visual comparison on a slice
start = 0
end = min(4000, len(eval_out))
v = eval_out.iloc[start:end].copy()

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

# Top panel: model confidence over time.
axes[0].plot(v["frame"], v["fight_probability"], lw=1)
axes[0].axhline(THRESHOLD, color="black", linestyle="--", label="threshold")
axes[0].set_ylabel("P(fight)")
axes[0].legend(loc="upper right")

# Bottom panel: manual vs model hard labels (0/1).
axes[1].plot(v["frame"], v["label"]+.1, label="manual", lw=1)
axes[1].plot(v["frame"], v["predicted_label"], label="predicted", lw=1, alpha=0.8)
axes[1].set_ylabel("Label")
axes[1].set_xlabel("Frame")
axes[1].legend(loc="upper right")

plt.suptitle(f"Minimal baseline: train={TRAIN_GT}, eval={EVAL_GT}")
plt.tight_layout()
plt.show()

## Quantitative evaluation on gt2 (after viewing the overlay)

Now that we visually inspected predicted vs manual labels, we compute numeric metrics to summarize performance.

## How to read the evaluation metrics

Before we evaluate the model, here are the four key metrics and what they mean.

- **Accuracy**: fraction of all frames classified correctly.
  - Quantitative: $(TP + TN) / (TP + TN + FP + FN)$
  - Intuition: "How often am I right overall?"

- **Precision**: fraction of predicted fight frames that are truly fights.
  - Quantitative: $TP / (TP + FP)$
  - Intuition: "When the model says 'fight', how trustworthy is that?"

- **Recall** (Sensitivity): fraction of true fight frames that were detected.
  - Quantitative: $TP / (TP + FN)$
  - Intuition: "How many real fights did I manage to catch?"

- **F1 score**: harmonic mean of precision and recall.
  - Quantitative: $2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$
  - Intuition: "Single balance score that is high only when both precision and recall are high."

Where:
- $TP$: true positives, $TN$: true negatives, $FP$: false positives, $FN$: false negatives.

In [ ]:
# Next: compute accuracy/precision/recall/F1 and compare gt2 with gt1 metrics.
# Metric definitions for binary classification:
# Accuracy  = (tp + tn) / (tp + tn + fp + fn)
# Precision = tp / (tp + fp)   -> among predicted fights, how many are real fights
# Recall    = tp / (tp + fn)   -> among real fights, how many we detected
# F1 score  = 2 * (precision * recall) / (precision + recall)

# Compute standard binary-classification metrics on gt2.
acc = accuracy_score(y_eval, pred_eval)
prec = precision_score(y_eval, pred_eval, zero_division=0)
rec = recall_score(y_eval, pred_eval, zero_division=0)
f1 = f1_score(y_eval, pred_eval, zero_division=0)

# Confusion matrix entries:
# tn=true negatives, fp=false positives, fn=false negatives, tp=true positives
tn, fp, fn, tp = confusion_matrix(y_eval, pred_eval, labels=[0, 1]).ravel()

metrics_df = pd.DataFrame([
    {
        "dataset": EVAL_GT,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }
])


# Side-by-side comparison with previously computed train metrics.
compare_metrics_df = pd.concat([train_metrics_df, metrics_df], ignore_index=True)
print("\nTrain vs eval metrics:")
display(compare_metrics_df)

## Why is performance lower on gt2 than on gt1?

This is expected and important for interpretation.

- The model was trained on `gt1`, so it can fit `gt1` patterns very closely (often giving very high train scores).
- `gt2` is unseen data and may differ in fish behavior, lighting, tracking noise, or recording conditions.
- The model only uses three simple frame-wise features, so it cannot capture all fight dynamics.
- The fixed threshold (`0.5`) may not be optimal for `gt2` even if it works well on `gt1`.

In short:
- **`gt1` score** tells us if the model can learn the training examples.
- **`gt2` score** tells us how well the model generalizes to new data (the more important test).

## Threshold Sweep on gt2

This shows how precision, recall, and F1 change when we vary the probability threshold.

In [ ]:
# Next: sweep decision thresholds to compare precision/recall/F1 and identify the best F1 threshold.
# Beginner note: lowering threshold usually increases recall and decreases precision.
thresholds = np.linspace(0.1, 0.9, 17)
rows = []

# Evaluate metrics for each threshold candidate.
for th in thresholds:
    pred_th = (prob_eval >= th).astype(int)
    rows.append({
        "threshold": th,
        "precision": precision_score(y_eval, pred_th, zero_division=0),
        "recall": recall_score(y_eval, pred_th, zero_division=0),
        "f1": f1_score(y_eval, pred_th, zero_division=0),
    })

# Put sweep results into a table.
sweep_df = pd.DataFrame(rows)
print("Threshold sweep preview:")
display(sweep_df.head())

# Find threshold with highest F1 in this sweep grid.
best_idx = sweep_df["f1"].idxmax()
best_row = sweep_df.loc[[best_idx]].copy()
print("Best threshold by F1:")
display(best_row)

# Plot precision/recall/F1 curves against threshold.
plt.figure(figsize=(10, 4))
plt.plot(sweep_df["threshold"], sweep_df["precision"], label="precision")
plt.plot(sweep_df["threshold"], sweep_df["recall"], label="recall")
plt.plot(sweep_df["threshold"], sweep_df["f1"], label="f1")
plt.axvline(THRESHOLD, color="black", linestyle="--", label="current threshold")
plt.axvline(float(best_row["threshold"].iloc[0]), color="red", linestyle=":", label="best F1 threshold")
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.title(f"Threshold sweep on {EVAL_GT} (model trained on {TRAIN_GT})")
plt.legend()
plt.tight_layout()
plt.show()

## Is this a valid minimal baseline?

Yes. This is a sensible first baseline because it is simple, fully interpretable, and cheap to run.

Expected limitations:
- No temporal context (no windowing), so brief motion noise can create false positives.
- Training on one recording (`gt1`) may not generalize well to different recording conditions in `gt2`.

Still, this gives a clear reference point before adding temporal features or broader training data.

## Apply the trained model to a new (very long) recording

> This section predicts fights on `new_dataset` using the same inline features as above (`inter_animal_distance`, `speed1`, `speed2`, `x1`, `y1`, `x2`, `y2`).

> Because this new recording has no manual labels in this workflow, we report prediction summaries and plots (not accuracy/precision/recall).

In [ ]:
# Next: load new dataset trajectories, compute the same inline features, and run model prediction.
# Path pattern mirrors Day5_randomForestFights.ipynb: DATA_ROOT/new_dataset_trajectories.csv
NEW_DATASET_ID = "new_dataset"

new_traj_path = os.path.join(DATA_ROOT, f"{NEW_DATASET_ID}_trajectories.csv")
new_traj_df = pd.read_csv(new_traj_path)

# Reuse the same inline feature function and same FEATURES list used for training.
new_feat_df = compute_minimal_features(new_traj_df, fps=FPS)

X_new = new_feat_df[FEATURES]
new_prob = clf.predict_proba(X_new)[:, 1]
new_pred = (new_prob >= THRESHOLD).astype(int)

new_out = new_feat_df[["frame"]].copy()
new_out["fight_probability"] = new_prob
new_out["predicted_label"] = new_pred

new_pred_path = os.path.join(DATA_ROOT, f"{NEW_DATASET_ID}_predictions_minimal.csv")
new_out.to_csv(new_pred_path, index=False)
print("Saved new-dataset predictions:", new_pred_path)

# Basic results (analogous style to gt2 section, but without label-based metrics).
print("\nNew dataset summary:")
print("Total frames:", len(new_out))
print("Predicted fight fraction:", float(new_out["predicted_label"].mean()))
print("Mean predicted probability:", float(new_out["fight_probability"].mean()))

print("\nPrediction preview:")
display(new_out.head(10))

# Visualization similar to gt2, but only model outputs are available.
start_new = 0
end_new = min(40000, len(new_out))
vn = new_out.iloc[start_new:end_new].copy()

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(vn["frame"], vn["fight_probability"], lw=1)
axes[0].axhline(THRESHOLD, color="black", linestyle="--", label="threshold")
axes[0].set_ylabel("P(fight)")
axes[0].legend(loc="upper right")

axes[1].plot(vn["frame"], vn["predicted_label"], label="predicted", lw=1, alpha=0.9)
axes[1].set_ylabel("Predicted label")
axes[1].set_xlabel("Frame")
axes[1].legend(loc="upper right")

plt.suptitle(f"Minimal baseline predictions on {NEW_DATASET_ID}")
plt.tight_layout()
plt.show()

In [ ]:
# Next: create a short annotated video chunk using the prediction file.
# Students can use this helper without seeing its internals.
from QuantBio_functions import annotate_most_fight_chunk

new_video_path = os.path.join(DATA_ROOT, f"{NEW_DATASET_ID}.mp4")
new_traj_path = os.path.join(DATA_ROOT, f"{NEW_DATASET_ID}_trajectories.csv")
annotated_new_video_path = os.path.join(DATA_ROOT, f"{NEW_DATASET_ID}_annotated_minimal.mp4")

annotate_most_fight_chunk(
    video_path=new_video_path,
    trajectory_csv=new_traj_path,
    predictions_csv=new_pred_path,
    output_path=annotated_new_video_path,
    window_minutes=5,
    threshold=THRESHOLD,
    trail_length=10,
 )

print("Saved annotated video:", annotated_new_video_path)